# S4.7 - Ranking functions
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** row_number, rank, dense_rank, ntile, percent_rank: pick the survivor row or the top N per group

In [0]:
# ============================================================
# S4.7 - Cell 2
# Ranking Functions:
# row_number, rank, dense_rank, ntile, percent_rank
#
# Goal:
# 1. Compare ranking functions on data containing ties.
# 2. Understand what each function does with tied values.
# 3. Understand why row_number needs a tie-breaker when
#    exactly one deterministic winner is required.
# 4. Compare Top-N using row_number, rank and dense_rank.
# 5. Implement deterministic latest-record-per-key logic,
#    completing the S4.5 deduplication pattern.
# 6. Validate latest-order-per-customer on samples.tpch.
#
# Scope:
# Ranking functions only.
#
# Deferred:
# - lag / lead / first / last          -> S4.8
# - running sum / rolling average      -> S4.9
# - rolling 7-day RetailPulse exercise -> S4.10
#
# Learning-output rule:
# Show enough intermediate information that the learner
# can explain WHY the ranking result happened.
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql import Window


print("=== S4.7: RANKING FUNCTIONS ===")
print()


# ============================================================
# Step 1 - Controlled Source Data
#
# Customer 1 deliberately has a tie:
#
# amount = 80
#
# for:
#
# order 102
# order 103
#
# This lets us compare how each ranking function treats ties.
# ============================================================

df = spark.createDataFrame(
    [
        (1, 101, 100),
        (1, 102, 80),
        (1, 103, 80),
        (1, 104, 50),

        (2, 201, 70),
        (2, 202, 60)
    ],
    """
    cust INT,
    order_id INT,
    amount INT
    """
)


print("=== STEP 1: SOURCE ===")


df.orderBy(
    "cust",
    F.col("amount").desc(),
    "order_id"
).show()

In [0]:
# ============================================================
# Step 2 - Compare Ranking Functions
#
# Window:
#
# partitionBy("cust")
# orderBy(amount DESC)
#
#
# Customer 1 amounts:
#
# 100
# 80
# 80
# 50
#
#
# Expected:
#
# ROW_NUMBER
#
# 1, 2, 3, 4
#
# Every row gets a unique number.
#
# The two amount=80 rows are tied on the ORDER BY value,
# so which one receives 2 or 3 is not guaranteed.
#
#
# RANK
#
# 1, 2, 2, 4
#
# Ties share a rank.
# The next rank has a gap.
#
#
# DENSE_RANK
#
# 1, 2, 2, 3
#
# Ties share a rank.
# No gap follows the tie.
#
#
# NTILE(2)
#
# Divides ordered rows into two row buckets.
#
# IMPORTANT:
#
# NTILE is not a "tie-preserving ranking".
# Equal values can be split across buckets.
#
#
# PERCENT_RANK
#
# Relative rank from 0.0 to 1.0.
#
# For partitions with more than one row:
#
# (rank - 1)
# --------------------
# (partition rows - 1)
# ============================================================

w_rank = (
    Window
    .partitionBy("cust")
    .orderBy(F.col("amount").desc()))

w_peer = (
    Window
    .partitionBy(
        "cust",
        "amount"))

ranked = (
    df

    # Show how many rows share the same ranking value.
    .withColumn(
        "peer_count",
        F.count("*").over(
            w_peer))
    
    .withColumn(
        "row_number",
        F.row_number().over(
            w_rank))
    
    .withColumn(
        "rank",
        F.rank().over(
            w_rank))

    .withColumn(
        "dense_rank",
        F.dense_rank().over(
            w_rank))

    .withColumn(
        "ntile2",
        F.ntile(2).over(
            w_rank))

    .withColumn(
        "percent_rank",
        F.round(
            F.percent_rank().over(
                w_rank),2)))

print()
print("=== STEP 2: RANKING FUNCTIONS ===")

print(
    """
HOW TO READ THIS

peer_count
→ how many rows have the same amount

row_number
→ unique position for every row

rank
→ ties share rank, gaps can appear

dense_rank
→ ties share rank, no gaps

ntile2
→ divides ordered ROWS into 2 buckets

percent_rank
→ relative rank between 0 and 1
"""
)


rank_rows = (
    ranked
    .orderBy(
        "cust",
        F.col("amount").desc(),
        "order_id"
    ).collect())


for r in rank_rows:

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"order={r['order_id']} | "
        f"amount={r['amount']} | "
        f"peers={r['peer_count']} | "
        f"row_number={r['row_number']} | "
        f"rank={r['rank']} | "
        f"dense_rank={r['dense_rank']} | "
        f"ntile2={r['ntile2']} | "
        f"percent_rank={r['percent_rank']}"
    )

In [0]:
# ============================================================
# Step 3 - Explain the Tie
#
# Customer 1:
#
# amount 100
# amount 80
# amount 80
# amount 50
#
# The two amount=80 rows form one peer group for:
#
# rank
# dense_rank
# percent_rank
#
# But row_number still has to assign two unique numbers.
# ============================================================

print()
print(
    "=== STEP 3: KEY OBSERVATION - TIES ==="
)


cust1_ranked = [
    r
    for r in rank_rows
    if r["cust"] == 1
]


for r in cust1_ranked:

    print(
        f"amount={r['amount']:<3} | "
        f"order={r['order_id']} | "
        f"row_number={r['row_number']} | "
        f"rank={r['rank']} | "
        f"dense_rank={r['dense_rank']}"
    )


print(
    """
Customer 1 ranking pattern:

amount 100
→ rank 1
→ dense_rank 1


amount 80
amount 80

→ BOTH rank 2
→ BOTH dense_rank 2


amount 50

RANK:
1, 2, 2, 4

Why 4?

Two rows occupied rank 2,
so rank 3 is skipped.


DENSE_RANK:
1, 2, 2, 3

No gap after the tie.


ROW_NUMBER:
always gives unique numbers.

For the tied amount=80 rows:

2 and 3

but without another ORDER BY column,
which row receives 2 is not guaranteed.
"""
)

In [0]:
# ============================================================
# Step 4 - Top N Per Customer
#
# Business question:
#
# "Give me the top 2 orders by amount for every customer."
#
#
# ROW_NUMBER <= 2
#
# → exactly 2 rows per customer
#
# BUT:
#
# if second place is tied, one tied row can be excluded.
#
#
# RANK <= 2
#
# → preserves all rows tied for positions 1 or 2
#
# Therefore customer 1 should return:
#
# amount 100
# amount 80
# amount 80
#
# → 3 rows
#
#
# DENSE_RANK <= 2
#
# → also returns the top 2 distinct amount levels
# → 3 rows for customer 1
# ============================================================

top2_row_number = (
    ranked
    .filter(
        F.col("row_number") <= 2
    )
)


top2_rank = (
    ranked
    .filter(
        F.col("rank") <= 2
    )
)


top2_dense_rank = (
    ranked
    .filter(
        F.col("dense_rank") <= 2
    )
)


print()
print(
    "=== STEP 4: TOP 2 PER CUSTOMER ==="
)


for cust in [1, 2]:

    rn_count = (
        top2_row_number
        .filter(
            F.col("cust") == cust
        )
        .count()
    )

    rank_count = (
        top2_rank
        .filter(
            F.col("cust") == cust
        )
        .count()
    )

    dense_count = (
        top2_dense_rank
        .filter(
            F.col("cust") == cust
        )
        .count()
    )


    print(
        f"RESULT cust={cust} | "
        f"row_number<=2 rows={rn_count} | "
        f"rank<=2 rows={rank_count} | "
        f"dense_rank<=2 rows={dense_count}"
    )


print()
print(
    "Customer 1 - ROW_NUMBER top 2:"
)


(
    top2_row_number
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "row_number"
    )
    .orderBy(
        "row_number"
    )
    .show()
)


print("Customer 1 - RANK top 2:")


(
    top2_rank
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "rank"
    )
    .orderBy(
        "rank",
        "order_id"
    ).show())


In [0]:
# ============================================================
# Step 5 - Deterministic Latest Record Per Key
#
# This completes an important idea from S4.5.
#
#
# Requirement:
#
# "Keep exactly ONE latest record per customer id."
#
#
# Controlled history:
#
# customer 10 has two records with the SAME latest timestamp.
#
# Therefore:
#
# orderBy(updated_at DESC)
#
# alone does NOT uniquely identify the winner.
#
#
# Add:
#
# version DESC
#
# as the business tie-breaker.
# ============================================================

history = spark.createDataFrame(
    [
        (
            10,
            "Asha v1",
            "2026-01-01 09:00:00",
            1
        ),
        (
            10,
            "Asha v2",
            "2026-02-01 09:00:00",
            1
        ),
        (
            10,
            "Asha v3",
            "2026-02-01 09:00:00",
            2
        ),
        (
            20,
            "Ben v1",
            "2026-01-15 10:00:00",
            1
        ),
        (
            20,
            "Ben v2",
            "2026-03-01 10:00:00",
            2
        )
    ],
    """
    id INT,
    name STRING,
    updated_at STRING,
    version INT
    """
)


history = (
    history
    .withColumn(
        "updated_at",
        F.to_timestamp(
            "updated_at"
        )
    )
)


print()
print(
    "=== STEP 5: LATEST RECORD PER KEY ==="
)


history.orderBy(
    "id",
    "updated_at",
    "version"
).show(
    truncate=False
)


# ------------------------------------------------------------
# Version A:
#
# updated_at only.
#
# Customer 10 has a timestamp tie.
#
# Exactly one row will still receive row_number=1,
# but which tied record wins is not guaranteed.
# ------------------------------------------------------------

w_latest_tied = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("updated_at").desc()
    )
)


latest_without_tiebreaker = (
    history
    .withColumn(
        "rn",
        F.row_number().over(
            w_latest_tied
        )
    )
)


print(
    "Without deterministic tie-breaker:"
)


(
    latest_without_tiebreaker
    .orderBy(
        "id",
        "rn"
    )
    .show(
        truncate=False
    )
)


# ------------------------------------------------------------
# Version B:
#
# updated_at DESC
# version DESC
#
# Now customer 10's winner is deterministic:
#
# Asha v3
# ------------------------------------------------------------

w_latest_deterministic = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("version").desc()
    )
)


latest_deterministic = (
    history
    .withColumn(
        "rn",
        F.row_number().over(
            w_latest_deterministic
        )
    )
)


latest_one_per_id = (
    latest_deterministic
    .filter(
        F.col("rn") == 1
    )
    .drop(
        "rn"
    )
)


print()
print(
    "With deterministic tie-breaker:"
)


latest_one_per_id.orderBy(
    "id"
).show(
    truncate=False
)


print(
    """
KEY LEARNING

row_number() always gives exactly one row number 1.

But:

exactly one
does NOT automatically mean
deterministically correct.

If ORDER BY values tie:

add a meaningful business tie-breaker.
"""
)


In [0]:
# ============================================================
# Step 6 - NTILE Learning Point
#
# NTILE divides ordered ROWS into buckets.
#
# It does NOT preserve ranking ties like rank/dense_rank.
#
# Customer 1 has four rows and NTILE(2):
#
# approximately:
#
# first 2 ordered rows  -> bucket 1
# next 2 ordered rows   -> bucket 2
#
# Because amount=80 is tied around that boundary,
# the two equal values can be placed into different buckets.
# ============================================================

print()
print(
    "=== STEP 6: NTILE AND TIES ==="
)


(
    ranked
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "ntile2"
    )
    .orderBy(
        "ntile2",
        F.col("amount").desc(),
        "order_id"
    )
    .show()
)


print(
    """
NTILE answers:

"Which row bucket does this row belong to?"

It does NOT answer:

"What competitive rank does this value have?"

Therefore do not use NTILE when tied values
must stay together.
"""
)


In [0]:
# ============================================================
# Step 7 - Real Data: Latest Order Per Customer
#
# Now "latest" really means time.
#
# Business rule:
#
# latest o_orderdate wins
#
# If multiple orders have the same latest date:
#
# highest o_orderkey wins as deterministic tie-breaker.
#
#
# Three customers from earlier exercises.
# ============================================================

o = (
    spark.table(
        "samples.tpch.orders"
    )
    .filter(
        F.col("o_custkey").isin(
            721180,
            382414,
            299701
        )
    )
    .select(
        "o_custkey",
        "o_orderkey",
        "o_orderdate",
        "o_totalprice"
    )
)


source_rows = (
    o.count()
)


source_customers = (
    o
    .select(
        "o_custkey"
    )
    .distinct()
    .count()
)


w_real = (
    Window
    .partitionBy(
        "o_custkey"
    )
    .orderBy(
        F.col(
            "o_orderdate"
        ).desc(),
        F.col(
            "o_orderkey"
        ).desc()
    )
)


ranked_orders = (
    o
    .withColumn(
        "rn",
        F.row_number().over(
            w_real
        )
    )
)


rows_after_window = (
    ranked_orders.count()
)


latest_orders = (
    ranked_orders
    .filter(
        F.col("rn") == 1
    )
)


latest_count = (
    latest_orders.count()
)


print()
print(
    "=== STEP 7: REAL DATA - LATEST ORDER ==="
)


print(
    f"RESULT source rows        : "
    f"{source_rows}"
)

print(
    f"RESULT rows after window  : "
    f"{rows_after_window}"
)

print(
    f"RESULT source customers   : "
    f"{source_customers}"
)

print(
    f"RESULT latest rows        : "
    f"{latest_count}"
)


print(
    "RESULT window kept source row count:",
    source_rows
    ==
    rows_after_window
)


print(
    "RESULT one latest row per customer:",
    latest_count
    ==
    source_customers
)


print()
print(
    "Latest order for each customer:"
)


for r in (
    latest_orders
    .orderBy(
        "o_custkey"
    )
    .collect()
):

    print(
        f"RESULT "
        f"cust={r['o_custkey']} | "
        f"order={r['o_orderkey']} | "
        f"date={r['o_orderdate']} | "
        f"total={r['o_totalprice']}"
    )

In [0]:
# ============================================================
# Step 8 - Controlled Checkpoint
# ============================================================

print()
print("=== STEP 8: S4.7 CHECKPOINT ===")


cust1 = (ranked.filter(F.col("cust") == 1))

cust1_rank_pattern = [
    (
        r["amount"],
        r["rank"],
        r["dense_rank"]
    )
    for r in (cust1.select("amount","rank","dense_rank")
        .orderBy(
            F.col("amount").desc()).collect())
]

print("RESULT customer 1 rank pattern:",cust1_rank_pattern)

print("RESULT latest deterministic rows:",latest_one_per_id.count())

print("RESULT real source rows preserved:",source_rows == rows_after_window)

print("RESULT real one-row-per-customer:",latest_count == source_customers)

# Controlled assertions.

assert (source_rows == rows_after_window)

assert (latest_count == source_customers)

assert (latest_one_per_id.count() == 2)

# Customer 10 deterministic winner should be Asha v3.

winner_10 = (
    latest_one_per_id
    .filter(F.col("id") == 10).first())

assert (winner_10["name"] == "Asha v3")

print("RESULT all controlled S4.7 checks passed.")


In [0]:
# ============================================================
# Step 9 - Final Learning Summary
# ============================================================

print()
print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
ROW_NUMBER

1, 2, 3, 4 ...

→ every row gets a unique position
→ ties do NOT share row numbers
→ useful when exactly one row must win


RANK

1, 2, 2, 4

→ ties share the same rank
→ gaps appear after ties


DENSE_RANK

1, 2, 2, 3

→ ties share the same rank
→ no gaps after ties


NTILE(n)

→ divides ordered rows into n buckets
→ based on row positions
→ tied values may be split across buckets


PERCENT_RANK

→ relative rank between 0 and 1

For partitions with more than one row:

(rank - 1)
--------------------
(partition rows - 1)


TOP N

row_number <= N

→ exactly N rows when enough rows exist
→ can split ties


rank <= N

→ keeps ties at the ranking boundary
→ may return more than N rows


dense_rank <= N

→ top N DISTINCT ranking values
→ may return more than N rows


DETERMINISTIC SURVIVOR

row_number() gives exactly one winner.

But if ORDER BY has ties:

the winner may not be deterministic.

Therefore use a meaningful tie-breaker.


Example:

updated_at DESC
version DESC


S4.5 CONNECTION

Do not blindly deduplicate.

First define the business survivor rule.

Then:

partitionBy(business_key)
orderBy(business_rule)
row_number()
filter rn = 1


CORE RULE

Choose the ranking function based on
the business meaning of ties.
""")


print(
    "=== S4.7 CELL 2 COMPLETED ==="
)

In [0]:
# ============================================================
# S4.7 - Cell 2
# Ranking Functions:
# row_number, rank, dense_rank, ntile, percent_rank
#
# Goal:
# 1. Compare ranking functions on data containing ties.
# 2. Understand what each function does with tied values.
# 3. Understand why row_number needs a tie-breaker when
#    exactly one deterministic winner is required.
# 4. Compare Top-N using row_number, rank and dense_rank.
# 5. Implement deterministic latest-record-per-key logic,
#    completing the S4.5 deduplication pattern.
# 6. Validate latest-order-per-customer on samples.tpch.
#
# Scope:
# Ranking functions only.
#
# Deferred:
# - lag / lead / first / last          -> S4.8
# - running sum / rolling average      -> S4.9
# - rolling 7-day RetailPulse exercise -> S4.10
#
# Learning-output rule:
# Show enough intermediate information that the learner
# can explain WHY the ranking result happened.
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql import Window


print("=== S4.7: RANKING FUNCTIONS ===")
print()


# ============================================================
# Step 1 - Controlled Source Data
#
# Customer 1 deliberately has a tie:
#
# amount = 80
#
# for:
#
# order 102
# order 103
#
# This lets us compare how each ranking function treats ties.
# ============================================================

df = spark.createDataFrame(
    [
        (1, 101, 100),
        (1, 102, 80),
        (1, 103, 80),
        (1, 104, 50),

        (2, 201, 70),
        (2, 202, 60)
    ],
    """
    cust INT,
    order_id INT,
    amount INT
    """
)


print(
    "=== STEP 1: SOURCE ==="
)


df.orderBy(
    "cust",
    F.col("amount").desc(),
    "order_id"
).show()


# ============================================================
# Step 2 - Compare Ranking Functions
#
# Window:
#
# partitionBy("cust")
# orderBy(amount DESC)
#
#
# Customer 1 amounts:
#
# 100
# 80
# 80
# 50
#
#
# Expected:
#
# ROW_NUMBER
#
# 1, 2, 3, 4
#
# Every row gets a unique number.
#
# The two amount=80 rows are tied on the ORDER BY value,
# so which one receives 2 or 3 is not guaranteed.
#
#
# RANK
#
# 1, 2, 2, 4
#
# Ties share a rank.
# The next rank has a gap.
#
#
# DENSE_RANK
#
# 1, 2, 2, 3
#
# Ties share a rank.
# No gap follows the tie.
#
#
# NTILE(2)
#
# Divides ordered rows into two row buckets.
#
# IMPORTANT:
#
# NTILE is not a "tie-preserving ranking".
# Equal values can be split across buckets.
#
#
# PERCENT_RANK
#
# Relative rank from 0.0 to 1.0.
#
# For partitions with more than one row:
#
# (rank - 1)
# --------------------
# (partition rows - 1)
# ============================================================

w_rank = (
    Window
    .partitionBy("cust")
    .orderBy(
        F.col("amount").desc()
    )
)


w_peer = (
    Window
    .partitionBy(
        "cust",
        "amount"
    )
)


ranked = (
    df

    # Show how many rows share the same ranking value.
    .withColumn(
        "peer_count",
        F.count("*").over(
            w_peer
        )
    )

    .withColumn(
        "row_number",
        F.row_number().over(
            w_rank
        )
    )

    .withColumn(
        "rank",
        F.rank().over(
            w_rank
        )
    )

    .withColumn(
        "dense_rank",
        F.dense_rank().over(
            w_rank
        )
    )

    .withColumn(
        "ntile2",
        F.ntile(2).over(
            w_rank
        )
    )

    .withColumn(
        "percent_rank",
        F.round(
            F.percent_rank().over(
                w_rank
            ),
            2
        )
    )
)


print()
print(
    "=== STEP 2: RANKING FUNCTIONS ==="
)


print(
    """
HOW TO READ THIS

peer_count
→ how many rows have the same amount

row_number
→ unique position for every row

rank
→ ties share rank, gaps can appear

dense_rank
→ ties share rank, no gaps

ntile2
→ divides ordered ROWS into 2 buckets

percent_rank
→ relative rank between 0 and 1
"""
)


rank_rows = (
    ranked
    .orderBy(
        "cust",
        F.col("amount").desc(),
        "order_id"
    )
    .collect()
)


for r in rank_rows:

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"order={r['order_id']} | "
        f"amount={r['amount']} | "
        f"peers={r['peer_count']} | "
        f"row_number={r['row_number']} | "
        f"rank={r['rank']} | "
        f"dense_rank={r['dense_rank']} | "
        f"ntile2={r['ntile2']} | "
        f"percent_rank={r['percent_rank']}"
    )


# ============================================================
# Step 3 - Explain the Tie
#
# Customer 1:
#
# amount 100
# amount 80
# amount 80
# amount 50
#
# The two amount=80 rows form one peer group for:
#
# rank
# dense_rank
# percent_rank
#
# But row_number still has to assign two unique numbers.
# ============================================================

print()
print(
    "=== STEP 3: KEY OBSERVATION - TIES ==="
)


cust1_ranked = [
    r
    for r in rank_rows
    if r["cust"] == 1
]


for r in cust1_ranked:

    print(
        f"amount={r['amount']:<3} | "
        f"order={r['order_id']} | "
        f"row_number={r['row_number']} | "
        f"rank={r['rank']} | "
        f"dense_rank={r['dense_rank']}"
    )


print(
    """
Customer 1 ranking pattern:

amount 100
→ rank 1
→ dense_rank 1


amount 80
amount 80

→ BOTH rank 2
→ BOTH dense_rank 2


amount 50

RANK:
1, 2, 2, 4

Why 4?

Two rows occupied rank 2,
so rank 3 is skipped.


DENSE_RANK:
1, 2, 2, 3

No gap after the tie.


ROW_NUMBER:
always gives unique numbers.

For the tied amount=80 rows:

2 and 3

but without another ORDER BY column,
which row receives 2 is not guaranteed.
"""
)


# ============================================================
# Step 4 - Top N Per Customer
#
# Business question:
#
# "Give me the top 2 orders by amount for every customer."
#
#
# ROW_NUMBER <= 2
#
# → exactly 2 rows per customer
#
# BUT:
#
# if second place is tied, one tied row can be excluded.
#
#
# RANK <= 2
#
# → preserves all rows tied for positions 1 or 2
#
# Therefore customer 1 should return:
#
# amount 100
# amount 80
# amount 80
#
# → 3 rows
#
#
# DENSE_RANK <= 2
#
# → also returns the top 2 distinct amount levels
# → 3 rows for customer 1
# ============================================================

top2_row_number = (
    ranked
    .filter(
        F.col("row_number") <= 2
    )
)


top2_rank = (
    ranked
    .filter(
        F.col("rank") <= 2
    )
)


top2_dense_rank = (
    ranked
    .filter(
        F.col("dense_rank") <= 2
    )
)


print()
print(
    "=== STEP 4: TOP 2 PER CUSTOMER ==="
)


for cust in [1, 2]:

    rn_count = (
        top2_row_number
        .filter(
            F.col("cust") == cust
        )
        .count()
    )

    rank_count = (
        top2_rank
        .filter(
            F.col("cust") == cust
        )
        .count()
    )

    dense_count = (
        top2_dense_rank
        .filter(
            F.col("cust") == cust
        )
        .count()
    )


    print(
        f"RESULT cust={cust} | "
        f"row_number<=2 rows={rn_count} | "
        f"rank<=2 rows={rank_count} | "
        f"dense_rank<=2 rows={dense_count}"
    )


print()
print(
    "Customer 1 - ROW_NUMBER top 2:"
)


(
    top2_row_number
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "row_number"
    )
    .orderBy(
        "row_number"
    )
    .show()
)


print(
    "Customer 1 - RANK top 2:"
)


(
    top2_rank
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "rank"
    )
    .orderBy(
        "rank",
        "order_id"
    )
    .show()
)


# ============================================================
# Step 5 - Deterministic Latest Record Per Key
#
# This completes an important idea from S4.5.
#
#
# Requirement:
#
# "Keep exactly ONE latest record per customer id."
#
#
# Controlled history:
#
# customer 10 has two records with the SAME latest timestamp.
#
# Therefore:
#
# orderBy(updated_at DESC)
#
# alone does NOT uniquely identify the winner.
#
#
# Add:
#
# version DESC
#
# as the business tie-breaker.
# ============================================================

history = spark.createDataFrame(
    [
        (
            10,
            "Asha v1",
            "2026-01-01 09:00:00",
            1
        ),
        (
            10,
            "Asha v2",
            "2026-02-01 09:00:00",
            1
        ),
        (
            10,
            "Asha v3",
            "2026-02-01 09:00:00",
            2
        ),
        (
            20,
            "Ben v1",
            "2026-01-15 10:00:00",
            1
        ),
        (
            20,
            "Ben v2",
            "2026-03-01 10:00:00",
            2
        )
    ],
    """
    id INT,
    name STRING,
    updated_at STRING,
    version INT
    """
)


history = (
    history
    .withColumn(
        "updated_at",
        F.to_timestamp(
            "updated_at"
        )
    )
)


print()
print(
    "=== STEP 5: LATEST RECORD PER KEY ==="
)


history.orderBy(
    "id",
    "updated_at",
    "version"
).show(
    truncate=False
)


# ------------------------------------------------------------
# Version A:
#
# updated_at only.
#
# Customer 10 has a timestamp tie.
#
# Exactly one row will still receive row_number=1,
# but which tied record wins is not guaranteed.
# ------------------------------------------------------------

w_latest_tied = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("updated_at").desc()
    )
)


latest_without_tiebreaker = (
    history
    .withColumn(
        "rn",
        F.row_number().over(
            w_latest_tied
        )
    )
)


print(
    "Without deterministic tie-breaker:"
)


(
    latest_without_tiebreaker
    .orderBy(
        "id",
        "rn"
    )
    .show(
        truncate=False
    )
)


# ------------------------------------------------------------
# Version B:
#
# updated_at DESC
# version DESC
#
# Now customer 10's winner is deterministic:
#
# Asha v3
# ------------------------------------------------------------

w_latest_deterministic = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("version").desc()
    )
)


latest_deterministic = (
    history
    .withColumn(
        "rn",
        F.row_number().over(
            w_latest_deterministic
        )
    )
)


latest_one_per_id = (
    latest_deterministic
    .filter(
        F.col("rn") == 1
    )
    .drop(
        "rn"
    )
)


print()
print(
    "With deterministic tie-breaker:"
)


latest_one_per_id.orderBy(
    "id"
).show(
    truncate=False
)


print(
    """
KEY LEARNING

row_number() always gives exactly one row number 1.

But:

exactly one
does NOT automatically mean
deterministically correct.

If ORDER BY values tie:

add a meaningful business tie-breaker.
"""
)


# ============================================================
# Step 6 - NTILE Learning Point
#
# NTILE divides ordered ROWS into buckets.
#
# It does NOT preserve ranking ties like rank/dense_rank.
#
# Customer 1 has four rows and NTILE(2):
#
# approximately:
#
# first 2 ordered rows  -> bucket 1
# next 2 ordered rows   -> bucket 2
#
# Because amount=80 is tied around that boundary,
# the two equal values can be placed into different buckets.
# ============================================================

print()
print(
    "=== STEP 6: NTILE AND TIES ==="
)


(
    ranked
    .filter(
        F.col("cust") == 1
    )
    .select(
        "order_id",
        "amount",
        "ntile2"
    )
    .orderBy(
        "ntile2",
        F.col("amount").desc(),
        "order_id"
    )
    .show()
)


print(
    """
NTILE answers:

"Which row bucket does this row belong to?"

It does NOT answer:

"What competitive rank does this value have?"

Therefore do not use NTILE when tied values
must stay together.
"""
)


# ============================================================
# Step 7 - Real Data: Latest Order Per Customer
#
# Now "latest" really means time.
#
# Business rule:
#
# latest o_orderdate wins
#
# If multiple orders have the same latest date:
#
# highest o_orderkey wins as deterministic tie-breaker.
#
#
# Three customers from earlier exercises.
# ============================================================

o = (
    spark.table(
        "samples.tpch.orders"
    )
    .filter(
        F.col("o_custkey").isin(
            721180,
            382414,
            299701
        )
    )
    .select(
        "o_custkey",
        "o_orderkey",
        "o_orderdate",
        "o_totalprice"
    )
)


source_rows = (
    o.count()
)


source_customers = (
    o
    .select(
        "o_custkey"
    )
    .distinct()
    .count()
)


w_real = (
    Window
    .partitionBy(
        "o_custkey"
    )
    .orderBy(
        F.col(
            "o_orderdate"
        ).desc(),
        F.col(
            "o_orderkey"
        ).desc()
    )
)


ranked_orders = (
    o
    .withColumn(
        "rn",
        F.row_number().over(
            w_real
        )
    )
)


rows_after_window = (
    ranked_orders.count()
)


latest_orders = (
    ranked_orders
    .filter(
        F.col("rn") == 1
    )
)


latest_count = (
    latest_orders.count()
)


print()
print(
    "=== STEP 7: REAL DATA - LATEST ORDER ==="
)


print(
    f"RESULT source rows        : "
    f"{source_rows}"
)

print(
    f"RESULT rows after window  : "
    f"{rows_after_window}"
)

print(
    f"RESULT source customers   : "
    f"{source_customers}"
)

print(
    f"RESULT latest rows        : "
    f"{latest_count}"
)


print(
    "RESULT window kept source row count:",
    source_rows
    ==
    rows_after_window
)


print(
    "RESULT one latest row per customer:",
    latest_count
    ==
    source_customers
)


print()
print(
    "Latest order for each customer:"
)


for r in (
    latest_orders
    .orderBy(
        "o_custkey"
    )
    .collect()
):

    print(
        f"RESULT "
        f"cust={r['o_custkey']} | "
        f"order={r['o_orderkey']} | "
        f"date={r['o_orderdate']} | "
        f"total={r['o_totalprice']}"
    )


# ============================================================
# Step 8 - Controlled Checkpoint
# ============================================================

print()
print(
    "=== STEP 8: S4.7 CHECKPOINT ==="
)


cust1 = (
    ranked
    .filter(
        F.col("cust") == 1
    )
)


cust1_rank_pattern = [
    (
        r["amount"],
        r["rank"],
        r["dense_rank"]
    )
    for r in (
        cust1
        .select(
            "amount",
            "rank",
            "dense_rank"
        )
        .orderBy(
            F.col("amount").desc()
        )
        .collect()
    )
]


print(
    "RESULT customer 1 rank pattern:",
    cust1_rank_pattern
)


print(
    "RESULT latest deterministic rows:",
    latest_one_per_id.count()
)


print(
    "RESULT real source rows preserved:",
    source_rows
    ==
    rows_after_window
)


print(
    "RESULT real one-row-per-customer:",
    latest_count
    ==
    source_customers
)


# Controlled assertions.

assert (
    source_rows
    ==
    rows_after_window
)


assert (
    latest_count
    ==
    source_customers
)


assert (
    latest_one_per_id.count()
    ==
    2
)


# Customer 10 deterministic winner should be Asha v3.

winner_10 = (
    latest_one_per_id
    .filter(
        F.col("id") == 10
    )
    .first()
)


assert (
    winner_10["name"]
    ==
    "Asha v3"
)


print(
    "RESULT all controlled S4.7 checks passed."
)


# ============================================================
# Step 9 - Final Learning Summary
# ============================================================

print()
print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
ROW_NUMBER

1, 2, 3, 4 ...

→ every row gets a unique position
→ ties do NOT share row numbers
→ useful when exactly one row must win


RANK

1, 2, 2, 4

→ ties share the same rank
→ gaps appear after ties


DENSE_RANK

1, 2, 2, 3

→ ties share the same rank
→ no gaps after ties


NTILE(n)

→ divides ordered rows into n buckets
→ based on row positions
→ tied values may be split across buckets


PERCENT_RANK

→ relative rank between 0 and 1

For partitions with more than one row:

(rank - 1)
--------------------
(partition rows - 1)


TOP N

row_number <= N

→ exactly N rows when enough rows exist
→ can split ties


rank <= N

→ keeps ties at the ranking boundary
→ may return more than N rows


dense_rank <= N

→ top N DISTINCT ranking values
→ may return more than N rows


DETERMINISTIC SURVIVOR

row_number() gives exactly one winner.

But if ORDER BY has ties:

the winner may not be deterministic.

Therefore use a meaningful tie-breaker.


Example:

updated_at DESC
version DESC


S4.5 CONNECTION

Do not blindly deduplicate.

First define the business survivor rule.

Then:

partitionBy(business_key)
orderBy(business_rule)
row_number()
filter rn = 1


CORE RULE

Choose the ranking function based on
the business meaning of ties.
""")


print(
    "=== S4.7 CELL 2 COMPLETED ==="
)

# S4.7 — Key Takeaways: Ranking Functions

## 1. Ranking Functions

```python
w = (
    Window
    .partitionBy("cust")
    .orderBy(F.col("amount").desc())
)
```

For values:

```text
100, 80, 80, 50
```

| Function | Result | Behaviour on ties |
|---|---|---|
| `row_number()` | 1, 2, 3, 4 | every row gets a unique number; tied rows need a tie-breaker for deterministic ordering |
| `rank()` | 1, 2, 2, 4 | ties share a rank; a gap follows |
| `dense_rank()` | 1, 2, 2, 3 | ties share a rank; no gap follows |
| `ntile(2)` | 1, 1, 2, 2 | divides ordered rows into approximately equal-sized buckets; tied values can be split |
| `percent_rank()` | 0.0, 0.33, 0.33, 1.0 | relative rank based on `(rank - 1) / (rows - 1)` |

### Easy way to remember

```text
row_number
→ unique row position

rank
→ ties + gaps

dense_rank
→ ties + no gaps

ntile
→ row buckets

percent_rank
→ relative ranking position
```

---

## 2. Top N Per Group

### row_number

```python
F.row_number().over(w) <= N
```

returns at most `N` rows per partition.

If the partition contains at least `N` rows:

```text
→ exactly N rows
```

Ties can be split.

---

### rank

```python
F.rank().over(w) <= N
```

keeps all rows whose competition rank is within the first `N` positions.

Ties at the boundary are preserved, so the result can contain more than `N` rows.

---

### dense_rank

```python
F.dense_rank().over(w) <= N
```

keeps the top `N` distinct ranked values.

Because ties are preserved, it can also return more than `N` rows.

### Observed for customer 1, Top 2

```text
row_number <= 2
→ 2 rows

rank <= 2
→ 3 rows

dense_rank <= 2
→ 3 rows
```

Customer 2 had no tie at the boundary:

```text
row_number → 2
rank       → 2
dense_rank → 2
```

---

## 3. rank vs dense_rank for Top N

These functions are similar but not identical.

Example:

```text
values:
100, 80, 80, 50
```

Ranks:

```text
rank:
1, 2, 2, 4

dense_rank:
1, 2, 2, 3
```

Therefore:

```text
rank <= 3
```

still returns only values ranked:

```text
1 and 2
```

because rank `3` does not exist.

But:

```text
dense_rank <= 3
```

returns the top three distinct values:

```text
100
80
50
```

So choose based on business meaning.

---

## 4. Keep One Deterministic Row Per Key

When the business requirement is:

```text
keep exactly one latest record per business key
```

a common pattern is:

```python
w = (
    Window
    .partitionBy("id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("version").desc()
    )
)

result = (
    df
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
)
```

### Important

`row_number() = 1` gives exactly one winner per partition.

But:

```text
exactly one winner
≠
deterministically correct winner
```

If the `ORDER BY` values tie, Spark is not required to choose the same business record as the winner.

Therefore add a meaningful business tie-breaker.

Examples:

```text
updated_at DESC
version DESC
event_sequence DESC
source_priority DESC
transaction_id DESC
```

The tie-breaker should represent the actual business rule.

---

## 5. Connection to S4.5 Deduplication

In S4.5 we learned:

```text
Do not blindly deduplicate just to reduce row counts.
```

S4.7 gives us the deterministic survivor pattern:

```text
1. Define business key
2. Define survivor rule
3. partitionBy(business_key)
4. orderBy(survivor_rule)
5. row_number()
6. keep rn = 1
```

This is preferable to blind `dropDuplicates()` when the business requires a specific record to survive.

---

## 6. Observed — Ranking Functions

Environment:

```text
Spark   : 4.2.0
Compute : Databricks Serverless
```

Customer 1:

```text
amounts:
100, 80, 80, 50
```

Observed:

```text
row_number
1, 2, 3, 4

rank
1, 2, 2, 4

dense_rank
1, 2, 2, 3

ntile(2)
1, 1, 2, 2

percent_rank
0.00, 0.33, 0.33, 1.00
```

The two `80` rows were peers for:

```text
rank
dense_rank
percent_rank
```

but `row_number()` still assigned them different row numbers.

---

## 7. Observed — Top 2

Customer 1:

```text
row_number <= 2
→ 2 rows

rank <= 2
→ 3 rows

dense_rank <= 2
→ 3 rows
```

This demonstrated that:

```text
row_number
→ can split a tie

rank / dense_rank
→ can preserve tied values
```

---

## 8. Observed — Latest Record Per Key

Controlled history contained a tie on:

```text
updated_at
```

for customer `10`.

Without another tie-breaker:

```text
Asha v2 → rn = 1
Asha v3 → rn = 2
```

For our intended business rule, `Asha v3` should survive because it has the higher version.

Therefore the first result was not the desired survivor.

After adding:

```text
updated_at DESC
version DESC
```

the deterministic survivors were:

```text
id 10 → Asha v3
id 20 → Ben v2
```

This validated the deterministic survivor pattern.

---

## 9. Observed — samples.tpch

Three customers were tested.

Source order counts:

```text
customer 299701 → 38
customer 382414 → 34
customer 721180 → 43

total → 115
```

Adding `row_number()` did not change the source row count:

```text
before window = 115
after window  = 115
```

Filtering:

```python
rn == 1
```

returned:

```text
3 rows
```

which matched:

```text
3 customers
```

### Latest orders observed

```text
299701
order 24894342
1998-05-23
182347.79

382414
order 14980806
1998-04-08
191007.44

721180
order 9019617
1998-04-16
210068.67
```

The business ordering used:

```text
o_orderdate DESC
o_orderkey DESC
```

so `o_orderkey` acted as the deterministic tie-breaker when multiple orders shared the same date.

---

## 10. What Was Not Verified

We observed one particular ordering for rows tied without a tie-breaker.

We did **not** verify that Spark changes the tied-row ordering between repeated executions.

The correct conclusion is:

```text
without a complete ordering,
the relative order of tied rows is not guaranteed
```

not:

```text
the result will definitely change every run
```

---

## 11. When to Use Which

### Exactly one deterministic winner

Use:

```text
row_number
+
complete business ordering
```

---

### Competition-style ranking

Use:

```text
rank
```

Example:

```text
1st
2nd
2nd
4th
```

---

### Ranking without gaps

Use:

```text
dense_rank
```

Example:

```text
1st
2nd
2nd
3rd
```

---

### Top N distinct values

Use:

```text
dense_rank <= N
```

---

### Top N ranking positions including ties

Use:

```text
rank <= N
```

---

### Exactly N rows when enough rows exist

Use:

```text
row_number <= N
```

---

### Buckets / quartiles / segments

Use:

```text
ntile(n)
```

Example:

```python
F.ntile(4)
```

for four approximately equal-sized row buckets.

Do not use `ntile()` when tied values must remain together.

---

## 12. Interview Q&A

### Q1. rank vs dense_rank?

`rank()` leaves gaps after ties:

```text
1, 2, 2, 4
```

`dense_rank()` does not:

```text
1, 2, 2, 3
```

---

### Q2. row_number vs rank?

`row_number()` always assigns a unique position.

`rank()` gives the same rank to rows tied on the ordering values.

---

### Q3. How do you keep the latest row per key?

Use:

```text
partitionBy(key)
+
orderBy(latest_column DESC, tie_breaker DESC)
+
row_number()
+
filter rn = 1
```

---

### Q4. Why add a tie-breaker?

Because the primary ordering column may contain ties.

Without a complete ordering, which tied row receives `row_number = 1` is not guaranteed.

---

### Q5. How do you return Top N including ties?

Usually use:

```text
rank <= N
```

when `N` refers to ranking positions.

Use:

```text
dense_rank <= N
```

when the requirement means the top `N` distinct ranked values.

---

### Q6. Can ntile split equal values?

Yes.

`ntile()` distributes ordered rows into buckets; it does not guarantee that equal values remain in the same bucket.

---

### Q7. Does adding a ranking window reduce the number of rows?

No.

The window itself keeps the original rows.

Rows are reduced only when we later filter, for example:

```python
F.col("rn") == 1
```

---

## 13. Core Understanding

```text
row_number
→ unique row position

rank
→ ties + gaps

dense_rank
→ ties + no gaps

ntile
→ row buckets

percent_rank
→ relative position
```

For deterministic survivor logic:

```text
business key
      ↓
partitionBy

business priority
      ↓
orderBy

unique winner
      ↓
row_number

keep winner
      ↓
rn = 1
```

### Core Rule

Choose the ranking function based on what **ties mean to the business**.

**S4.7 — Ranking Functions COMPLETED.**